In [1]:
import pandas as pd
import numpy as np

In [35]:
df = pd.read_csv("../data/matches.csv")

In [36]:
df.shape

print(df.columns)

Index(['Unnamed: 0', 'date', 'time', 'comp', 'round', 'day', 'venue', 'result',
       'gf', 'ga', 'opponent', 'xg', 'xga', 'poss', 'attendance', 'captain',
       'formation', 'referee', 'match report', 'notes', 'sh', 'sot', 'dist',
       'fk', 'pk', 'pkatt', 'season', 'team'],
      dtype='str')


In [16]:
# View mancity results in 2022

mancity = df[(df['team'] == 'Manchester City') & (df['date'].str.contains('2022'))]
print(mancity.shape)

mancity.to_csv("mancity.csv")

(34, 28)


In [17]:
# View results on 2022-05-22

datematch = df[df['date'].str.contains('2022-05-22')]
print(datematch.shape)

datematch.to_csv("datematch.csv")

(20, 28)


- arrange date in order
- add previous match stats (use rolling window)


Columns needed:

date, season, round, team, opponent, venue, prev_result, prev_gf, prev_sot, prev_xg, prev_ga, prev_poss, prev_opponnet, goals

<br>
Potantially important columns:

prev_gf, prev_sot, prev_xg

In [37]:
# Rearrange colums
df = df[['Unnamed: 0', 'date', 'season', 'round', 'team', 'opponent', 'venue', 'result',
       'gf', 'xg', 'sh', 'sot' ,'ga', 'xga', 'poss', 'formation', 'attendance', 'captain',
       'referee', 'match report', 'notes', 'dist', 'fk', 'pk', 'pkatt']]

In [38]:
# Arrange date in ascending order


df_sorted = df.sort_values(by='date', ascending=True)
df_sorted.to_csv("df_sorted.csv")

In [39]:
# create new dataframe with values from previous match for each team
df_sorted['prev_result'] = df_sorted.groupby('team')['result'].shift(1)
df_sorted['prev_gf'] = df_sorted.groupby('team')['gf'].shift(1)
df_sorted['prev_xg'] = df_sorted.groupby('team')['xg'].shift(1)
df_sorted['prev_sh'] = df_sorted.groupby('team')['sh'].shift(1)
df_sorted['prev_sot'] = df_sorted.groupby('team')['sot'].shift(1)
df_sorted['prev_opponent'] = df_sorted.groupby('team')['opponent'].shift(1)

In [40]:
print(df_sorted.columns)

Index(['Unnamed: 0', 'date', 'season', 'round', 'team', 'opponent', 'venue',
       'result', 'gf', 'xg', 'sh', 'sot', 'ga', 'xga', 'poss', 'formation',
       'attendance', 'captain', 'referee', 'match report', 'notes', 'dist',
       'fk', 'pk', 'pkatt', 'prev_result', 'prev_gf', 'prev_xg', 'prev_sh',
       'prev_sot', 'prev_opponent'],
      dtype='str')


In [ ]:
df_sorted.to_csv("df_sorted.csv")

In [7]:
import pandas as pd
import requests

# Example using football-data.org (v4 REST API)
API_KEY = "434b722b08a642a8a33a5785d565de76"
url = "https://api.football-data.org/v4/competitions/PL/matches"

headers = {"X-Auth-Token": API_KEY}

# Optional parameters to filter by season (e.g., 2024 season)
params = {"season": 2024}

response = requests.get(url, headers=headers, params=params)

if response.status_code == 200:
  data = response.json()
  matches = data.get("matches", [])

  # View columns in the first match to understand structure
  print(matches[0].keys())

  # Extract key fields into a clean DataFrame
  match_list = []
  for m in matches:
    match_list.append({
        "Matchday": m.get("matchday"),
        "Date": m.get("utcDate"),
        "HomeTeam": m["homeTeam"]["name"],
        "AwayTeam": m["awayTeam"]["name"],
        "HomeScore": m["score"]["fullTime"]["home"],
        "AwayScore": m["score"]["fullTime"]["away"],
    })

  df = pd.DataFrame(match_list)
  print(df.shape)
else:
  print(f"Error: {response.status_code} - {response.text}")

dict_keys(['area', 'competition', 'season', 'id', 'utcDate', 'status', 'matchday', 'stage', 'group', 'lastUpdated', 'homeTeam', 'awayTeam', 'score', 'odds', 'referees'])
(380, 6)
